# Basic Retrieval System

## Load Data

In [13]:
import pandas as pd

from load import LoadDataCSV

# Load datasets

df_loader=LoadDataCSV()

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

# Load text fields for training
post_texts = df_posts['text'].tolist()
fact_check_texts = df_fact_checks['claim'].apply(lambda x: x[0] if isinstance(x, tuple) else x).tolist()


In [8]:
# Check unique values in fact_check_post_mapping post_id
print(df_fact_check_post_mapping['post_id'].nunique())
print(df_fact_check_post_mapping['fact_check_id'].nunique())
print(len(df_fact_check_post_mapping))

21988
14223
25743


## Split in train and dev

In [14]:
# Divide the df_fact_check_post_mapping into training and development sets
from sklearn.model_selection import train_test_split

def split_by_id(df):
    # Get unique IDs and split into train/test
    unique_ids = df['post_id'].unique()
    train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

    # Create training and test sets
    train_set = df[df['post_id'].isin(train_ids)]
    test_set = df[df['post_id'].isin(test_ids)]

    return train_set, test_set

df_fact_check_post_mapping_train, df_fact_check_post_mapping_dev = split_by_id(df_fact_check_post_mapping)

In [24]:
df_fact_check_post_mapping_train

,post_id,fact_check_id
0,2228,33
1,2228,23568
2,2228,194577
3,2229,33
4,2229,23568
...,...,...
25737,14764,201987
25738,14765,201987
25739,17076,201987
25740,18203,201987


# Retrieval

In [27]:
from sentence_transformers import SentenceTransformer, util
import torch

In [2]:

# Load the model to create embeddings
model = SentenceTransformer("dunzhang/stella_en_400M_v5", trust_remote_code=True)

# Save the model locally
model.save("stella_en_400M_v5")

Some weights of the model checkpoint at dunzhang/stella_en_400M_v5 were not used when initializing NewModel: ['new.pooler.dense.bias', 'new.pooler.dense.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [26]:
# Load the model from the local directory
model = SentenceTransformer("/Users/santiagolares/Projects/semeval_2025/stella_en_400M_v5", device='cpu',trust_remote_code=True)

In [ ]:
import chromadb

# Configuration of ChromaDB and SentenceTransformer
client = chromadb.Client()


Some weights of the model checkpoint at dunzhang/stella_en_400M_v5 were not used when initializing NewModel: ['new.pooler.dense.bias', 'new.pooler.dense.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [28]:

# Function to split text into chunks
def chunk_text(text, max_tokens=512):
    sentences = text.split(". ")
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Load claims into the vector database with their embeddings
df_fact_checks['claim_embedding'] = df_fact_checks['claim'].apply(lambda x: embedding_fn.encode(x, convert_to_tensor=False))

for _, row in df_fact_checks.iterrows():
    client.add(
        text=row['claim'],
        embedding=row['claim_embedding'].cpu().numpy(),  # convert to numpy if necessary
        metadata={'fact_check_id': row['fact_check_id']}
    )

# Function to search for relevant claims by chunks
def find_relevant_claims_by_chunk(text, top_k=3, threshold=0.5):
    chunks = chunk_text(text)
    results = []
    for chunk in chunks:
        chunk_embedding = embedding_fn.encode(chunk, convert_to_tensor=True).to('mps')
        # Query the vector database with each chunk
        top_claims = client.query(embedding=chunk_embedding.cpu().numpy(), top_k=top_k)
        
        for claim in top_claims:
            if claim['distance'] >= threshold:
                results.append({
                    'claim': claim['text'],
                    'fact_check_id': claim['metadata']['fact_check_id'],
                    'similarity': claim['distance']
                })
    
    # Filter and group the top-k global results among all chunks
    results_df = pd.DataFrame(results)
    top_results = results_df.nlargest(top_k, 'similarity')
    return top_results

# Execution on each long text in df_posts
final_results = []
for index, row in df_posts.iterrows():
    relevant_claims = find_relevant_claims_by_chunk(row['text'])
    for _, claim_row in relevant_claims.iterrows():
        final_results.append({
            'post_id': row['post_id'],
            'text': row['text'],
            'claim': claim_row['claim'],
            'fact_check_id': claim_row['fact_check_id'],
            'similarity': claim_row['similarity']
        })

# Convert final results to a DataFrame
final_results_df = pd.DataFrame(final_results)
print(final_results_df[['post_id', 'text', 'claim', 'fact_check_id', 'similarity']])


NotImplementedError: No operator found for `memory_efficient_attention_forward` with inputs:
     query       : shape=(1, 33, 16, 64) (torch.float32)
     key         : shape=(1, 33, 16, 64) (torch.float32)
     value       : shape=(1, 33, 16, 64) (torch.float32)
     attn_bias   : <class 'xformers.ops.fmha.attn_bias.BlockDiagonalMask'>
     p           : 0.0
`ckF` is not supported because:
    device=cpu (supported: {'cuda'})
    dtype=torch.float32 (supported: {torch.bfloat16, torch.float16})
    operator wasn't built - see `python -m xformers.info` for more info

In [25]:
import numpy as np
from tqdm import tqdm

# Cargar el modelo en CPU
model=embedding_fn

# Crear una función para procesar en lotes
def batch_encode(texts, model, batch_size=32):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_embeddings = model.encode(batch, convert_to_tensor=False)  # Mantener como arrays de NumPy
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)

# Aplicar la función en el DataFrame
df_fact_checks['claim_embedding'] = batch_encode(df_fact_checks['claim'].tolist(), model)


  0%|          | 0/4805 [00:00<?, ?it/s]


NotImplementedError: No operator found for `memory_efficient_attention_forward` with inputs:
     query       : shape=(1, 2734, 16, 64) (torch.float32)
     key         : shape=(1, 2734, 16, 64) (torch.float32)
     value       : shape=(1, 2734, 16, 64) (torch.float32)
     attn_bias   : <class 'xformers.ops.fmha.attn_bias.BlockDiagonalMask'>
     p           : 0.0
`ckF` is not supported because:
    device=cpu (supported: {'cuda'})
    dtype=torch.float32 (supported: {torch.bfloat16, torch.float16})
    operator wasn't built - see `python -m xformers.info` for more info

In [ ]:
# Agregar las metricas de performance que se usa en la evaluacion